<a href="https://colab.research.google.com/github/jjcodes999/flyrank-ml-repo/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jjcodes999/flyrank-ml-repo/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Answer:

The paper (*The State of AI-Driven SEO in Numbers*, March 2026; 341,701 pages, 57 brands) leads with direct aggregate comparisons, places ML in an exploratory appendix, and states that it is observational. My questions below are suggestions for the next level of rigor, the same questions I then apply to my own model.

### Finding A: "Click Capture by Position Tier" (Finding #3, p. 8)
**The claim:** weighted CTR (total clicks ÷ total impressions per tier) falls as pages move down the results: Top 3 0.423%, Page 1 (4–10) 0.339%, Striking (11–20) 0.325%, Page 3–5 0.163%, Deep 0.050%. The paper already improved on an older per-row average, and says these are not generic Google benchmarks.

**Where the numbers come from:** there is no model label here. It is a descriptive ratio pooled across all 57 brands, with each page placed in a tier by its average position over the 90-day window.

**My methodology questions:**
1. **Does the pattern hold inside each brand?** Because clicks and impressions are pooled, the largest brands carry the most weight in every tier. Showing the median of per-brand CTRs next to the pooled number, plus the largest brand's share of impressions per tier, would show whether the curve is a portfolio-wide pattern or driven by a few big brands. (In my own March data, one client filled about half of my top-100 list, so this risk is real; my check is below.)
2. **Is Page 1 vs Striking a real difference?** 0.339% and 0.325% are very close, and the paper reports no confidence intervals (p. 36). Page counts per tier and an uncertainty range would tell the reader whether these two tiers actually differ.
3. **How do the definitions compare?** The repo's data dictionary quotes about 2.78% CTR for positions 1–3 at warehouse scale, while the paper's Top-3 figure is 0.423%. A one-line note on what differs (population, window, or how position is averaged) would help readers reconcile them.

### Finding B: "What Predicts Growth?" (ML appendix, p. 29)
**The claim:** a logistic regression separates growing from declining pages with 71% holdout accuracy. The paper correctly calls this descriptive and exploratory.

**Where the label comes from:** trend direction, computed from impressions in the last 30 days versus the 30 days before (Up / Down = more than 10% change, p. 5). The model uses an 80/20 split (p. 36) on 61.8K active pages.

**My methodology questions:**
1. **Do any features overlap the label window?** If features such as impressions, clicks or days visible are 90-day totals, they include the same 30 days the label is computed from, so the model could partly read the outcome. Building features only from the days *before* the label window would remove that risk.
2. **What is the naive baseline?** Finding #1 counts 74,187 rising vs 45,272 falling pages (about 62% rising). If the ML sample has a similar mix, always guessing "growing" would already reach about 62%, so 71% would be roughly 9 points of skill. Printing the majority-class accuracy next to 71% would make the result easy to judge.
3. **Were brands kept separate between training and testing?** An 80/20 split of pages can place pages from the same brand on both sides, which can make accuracy look higher than it would be on a new brand. A brand-grouped split would show how well the model carries over to brands it has not seen.

In [1]:
%pip -q install duckdb

import duckdb
import numpy as np
import pandas as pd
import sklearn
from google.colab import userdata

# Connect to the warehouse
HF_TOKEN = userdata.get("HF_TOKEN")
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = "hf://datasets/FlyRank/internship-warehouse"
MARCH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

# Same frame as ML-04 / ML-07 / ML-08: one row per page. Clues 1-15 March, answer 16-31 March.
pages = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
        SUM(gsc_impressions) FILTER (WHERE report_date <= DATE '2026-03-15') AS impressions_h1,
        SUM(gsc_clicks)      FILTER (WHERE report_date <= DATE '2026-03-15') AS clicks_h1,
        SUM(gsc_avg_position * gsc_impressions) FILTER (WHERE report_date <= DATE '2026-03-15' AND gsc_avg_position > 0)
          / NULLIF(SUM(gsc_impressions) FILTER (WHERE report_date <= DATE '2026-03-15' AND gsc_avg_position > 0), 0) AS position_h1,
        COUNT(DISTINCT report_date) FILTER (WHERE report_date <= DATE '2026-03-15' AND gsc_impressions > 0) AS days_visible_h1,
        SUM(gsc_impressions) FILTER (WHERE report_date > DATE '2026-03-15') AS impressions_h2,
        SUM(gsc_clicks)      FILTER (WHERE report_date > DATE '2026-03-15') AS clicks_h2,
        SUM(gsc_avg_position * gsc_impressions) FILTER (WHERE report_date > DATE '2026-03-15' AND gsc_avg_position > 0)
          / NULLIF(SUM(gsc_impressions) FILTER (WHERE report_date > DATE '2026-03-15' AND gsc_avg_position > 0), 0) AS position_h2
    FROM {MARCH}
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
    HAVING impressions_h1 >= 100 AND impressions_h2 >= 50
""").df()
pages = pages.dropna(subset=["position_h1", "position_h2"]).copy()
pages = pages.sort_values(["client_hash_id", "content_hash_id"]).reset_index(drop=True)  # fixed row order -> reproducible

bands = [0, 3, 5, 7, 10, 15, 20, 30, 50, 1000]
pages["ctr_h1"] = pages["clicks_h1"] / pages["impressions_h1"] * 100
pages["band_h1"] = pd.cut(pages["position_h1"], bands)
pages["expected_ctr_h1"] = pages.groupby("band_h1", observed=True)["ctr_h1"].transform("median")
pages["missed_clicks_h1"] = pages["impressions_h1"] * (pages["expected_ctr_h1"] - pages["ctr_h1"]) / 100

pages["ctr_h2"] = pages["clicks_h2"] / pages["impressions_h2"] * 100
pages["band_h2"] = pd.cut(pages["position_h2"], bands)
pages["expected_ctr_h2"] = pages.groupby("band_h2", observed=True)["ctr_h2"].transform("median")
pages["still_under"] = (pages["ctr_h2"] < pages["expected_ctr_h2"]).astype(int)

# Frozen Week-4 baseline rule
MIN_IMPRESSIONS, MAX_POSITION = 1000, 20
flagged = (pages["impressions_h1"] >= MIN_IMPRESSIONS) & (pages["position_h1"] <= MAX_POSITION) & (pages["ctr_h1"] < pages["expected_ctr_h1"])
pages["baseline_score"] = np.where(flagged, pages["missed_clicks_h1"], 0.0)
FEATURES = ["impressions_h1", "ctr_h1", "position_h1", "days_visible_h1", "missed_clicks_h1"]

print("Pages:", len(pages), "| clients:", pages["client_hash_id"].nunique(),
      "| base rate:", round(pages["still_under"].mean(), 2), "| scikit-learn", sklearn.__version__)

# CHECK OF QUESTION A1 ON MY OWN DATA: pooled CTR vs the typical client's CTR, per position tier (1-15 March)
tier = pd.cut(pages["position_h1"], [0, 3, 10, 20, 50, 1000],
              labels=["Top 3 (1-3)", "Page 1 (4-10)", "Striking (11-20)", "Page 3-5 (21-50)", "Deep (50+)"])
by_client = (pages.assign(tier=tier)
             .groupby(["tier", "client_hash_id"], observed=True)
             .agg(clicks=("clicks_h1", "sum"), impressions=("impressions_h1", "sum")))
by_client["ctr"] = by_client["clicks"] / by_client["impressions"] * 100
g = by_client.groupby("tier", observed=True)
check = pd.DataFrame({
    "pooled_weighted_ctr_pct": g["clicks"].sum() / g["impressions"].sum() * 100,
    "median_client_ctr_pct": g["ctr"].median(),
    "clients": g.size(),
    "biggest_client_share_of_impressions": g["impressions"].apply(lambda s: s.max() / s.sum()),
}).round(3)
display(check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Pages: 75817 | clients: 37 | base rate: 0.35 | scikit-learn 1.6.1


,pooled_weighted_ctr_pct,median_client_ctr_pct,clients,biggest_client_share_of_impressions
tier,,,,
Top 3 (1-3),0.437,0.500,23,0.407
Page 1 (4-10),0.330,0.368,37,0.372
Striking (11-20),0.336,0.245,32,0.367
Page 3-5 (21-50),0.141,0.130,26,0.726
Deep (50+),0.038,0.033,21,0.446


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.